# 6차시 실습 · 머신러닝 기초 — 데이터로 배우는 컴퓨터
> 데이터분석 6단계 중 **⑤ 분석·예측** — 지금까지 탐색한 데이터로 '예측하는 모델'을 만든다

**머신러닝 기본 흐름** — ① 문제 정의 → ② 데이터 준비(X·y) → ③ 학습용·평가용 나누기 → ④ 모델 학습(`fit`) → ⑤ 예측(`predict`) → ⑥ 평가

**진행 방법 (PRIMM)** — 🔮 **예측** → ▶️ **실행** → 👀 **관찰** → 🔧 **수정** → ✏️ **직접 만들기** / ✍️ **해석 쓰기**

**사용 데이터** (`data/` 폴더)
| 파일 | 내용 | 출처 |
|---|---|---|
| (코드 안) | 기온과 아이스크림 판매량 7일 | 교재 6장 예제 |
| `diamonds.csv` | 다이아몬드 53,940개의 무게·크기·품질·가격 | seaborn 내장 데이터 (교재 6장) |
| `wine.csv` | 와인 178병의 성분 13가지 + 실제 품종 | scikit-learn 내장 데이터 (교재 6장) |
| `bike_2025.csv` | 공공자전거 일별 대여 365일 (1·4차시, 가상 데이터) | 수업 연습용 |

## 0. 준비

도구를 불러온다. scikit-learn(`sklearn`)은 파이썬의 대표 머신러닝 라이브러리로, 코랩에는 이미 설치되어 있다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib
import sklearn

plt.rcParams['figure.figsize'] = (6, 3.5)
DATA = 'data' if os.path.exists('data') else '../data'
print('scikit-learn 버전:', sklearn.__version__)

---
# PART A. 지도학습 · 회귀 ① — 기온으로 아이스크림 판매량 예측
**문제** 기온(℃)을 알면 아이스크림이 몇 개 팔릴지 예측할 수 있을까?
- 문제(입력) **X** = 기온 · 정답(출력) **y** = 판매량 · 정답이 있는 데이터로 배우므로 **지도학습**, 정답이 숫자이므로 **회귀**

▶️ 교재 6장의 7일치 데이터를 만들고 산점도로 관계를 본다.

In [ ]:
df_ice = pd.DataFrame({
    '기온': [18, 20, 22, 24, 26, 28, 30],
    '아이스크림_판매량': [42, 47, 55, 58, 66, 72, 78]
})
plt.scatter(df_ice['기온'], df_ice['아이스크림_판매량'])
plt.title('기온과 아이스크림 판매량')
plt.xlabel('기온(℃)')
plt.ylabel('판매량(개)')
plt.show()

✏️ 빈칸 채우기: X는 **표(DataFrame)** 모양이라 대괄호 두 겹, y는 **한 줄(Series)** 이라 한 겹으로 고른다.

In [ ]:
X = df_ice[___]
y = df_ice['아이스크림_판매량']
print(type(X), X.shape)
print(type(y), y.shape)

▶️ 모델을 만들고(`LinearRegression()`), 데이터로 학습시킨다(`fit`).

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_ice[['기온']]
y = df_ice['아이스크림_판매량']
model = LinearRegression()
model.fit(X, y)

🔮 예측 → 실행: 기온이 32℃인 날은 몇 개 팔릴까? (예측: 약 ___개)

In [ ]:
new_X = pd.DataFrame({'기온': [32]})
model.predict(new_X)

👀 관찰: 모델이 배운 것은 **직선 하나**다. 기울기(`coef_`)와 절편(`intercept_`)을 확인하고 직선을 그려 본다.

In [ ]:
print('기울기:', model.coef_[0].round(2), '→ 기온이 1℃ 오르면 판매량이 약', model.coef_[0].round(1), '개 늘어난다')
print('절편:', model.intercept_.round(2))
xs = pd.DataFrame({'기온': range(16, 35)})
plt.scatter(df_ice['기온'], df_ice['아이스크림_판매량'], label='실제 데이터')
plt.plot(xs['기온'], model.predict(xs), color='tomato', label='모델이 배운 직선')
plt.scatter([32], model.predict(new_X), color='tomato', s=80, marker='*', label='32℃ 예측')
plt.legend()
plt.title('선형 회귀 = 데이터를 가장 잘 지나는 직선 찾기')
plt.xlabel('기온(℃)')
plt.ylabel('판매량(개)')
plt.show()

✏️ 빈칸 채우기: 0℃인 날을 예측해 보자. 결과가 말이 되는가?

In [ ]:
model.predict(pd.DataFrame({'기온': [___]}))

✍️ **해석 쓰기** — 0℃의 예측값이 음수(약 −13개)가 나온 이유는? 모델이 배운 데이터의 범위(18~30℃) 밖을 예측할 때 무엇을 조심해야 할까?

> (여기에 한두 문장으로 적어 보세요)

---
# PART B. 지도학습 · 회귀 ② — 다이아몬드 가격 예측
**문제** 다이아몬드의 크기 정보(캐럿·가로·세로·높이)만으로 가격을 예측할 수 있을까?

▶️ 데이터를 불러와 구조를 확인한다. (교재는 `sns.load_dataset('diamonds')` — 같은 데이터를 파일로 저장해 두었다)

In [ ]:
dia = pd.read_csv(f'{DATA}/diamonds.csv')
print(dia.shape)
dia.head()

| 열 | 뜻 | 열 | 뜻 |
|---|---|---|---|
| carat | 무게(캐럿) | price | 가격(미국 달러) |
| cut | 컷 등급 (Fair < Good < Very Good < Premium < Ideal) | x · y · z | 가로·세로·높이(mm) |
| color | 색상 등급 (D가 최고 ~ J) | depth · table | 비율 정보(%) |
| clarity | 선명도 (I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF) | | |

✏️ 빈칸 채우기: 수치형 열의 상관계수를 히트맵으로 그려, 가격과 가장 관계가 깊은 열을 찾는다. (4차시)

In [ ]:
corr = dia.corr(numeric_only=True).round(2)
sns.heatmap(corr, annot=True, cmap=___)
plt.title('다이아몬드 수치형 열의 상관관계')
plt.show()

▶️ 캐럿과 가격의 산점도

In [ ]:
sns.scatterplot(data=dia, x='carat', y='price', s=5, alpha=0.3)
plt.title('캐럿과 가격의 관계')
plt.show()

## 학습용 / 평가용 나누기 — 시험 문제를 미리 보면 안 된다

✏️ 빈칸 채우기: 70%는 학습용, 30%는 평가용으로 나눈다. `random_state`를 고정하면 매번 같게 나뉜다.

In [ ]:
from sklearn.model_selection import train_test_split

X = dia[['carat', 'x', 'y', 'z']]
y = dia['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('학습용:', len(X_train), '/ 평가용:', ___)

✏️ 빈칸 채우기: 학습용 데이터로만 모델을 학습시키고, 평가용 데이터로 예측한다.

In [ ]:
model = LinearRegression()
model.___(X_train, y_train)
y_pred = model.___(X_test)
y_pred[:5].round(0)

▶️ 실제 가격과 예측 가격을 나란히 비교한다.

In [ ]:
model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)
compare = X_test.iloc[:5].copy()
compare['실제 가격'] = y_test.iloc[:5].values
compare['예측 가격'] = y_pred[:5].round(0)
compare

## 평가 — 얼마나 잘 맞히나?

✏️ 빈칸 채우기: MAE(평균 절대 오차)와 R²(결정계수)를 구한다.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = ___(y_test, y_pred)
r2 = ___(y_test, y_pred)
print(f'MAE: {mae:.0f}달러  → 평균적으로 이만큼 틀린다')
print(f'R²: {r2:.3f}      → 1에 가까울수록 가격의 변화를 잘 설명한다')

🔮 예측 → 실행: '모든 다이아몬드를 평균 가격이라고 찍는' 엉터리 방법의 MAE는 얼마일까? 모델과 비교한다.

In [ ]:
guess = np.full(len(y_test), y_train.mean())
print('평균으로 찍기 MAE:', round(mean_absolute_error(y_test, guess)), '달러')
print('선형 회귀 모델 MAE:', round(mean_absolute_error(y_test, y_pred)), '달러')

👀 관찰: 실제 가격(x축) vs 예측 가격(y축). 빨간 대각선 위에 있으면 정확히 맞힌 것이다.

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(y_test, y_pred, s=3, alpha=0.3)
plt.plot([0, 20000], [0, 20000], color='red')
plt.xlim(0, 20000)
plt.ylim(-3000, 20000)
plt.title('실제 가격 vs 예측 가격')
plt.xlabel('실제 가격')
plt.ylabel('예측 가격')
plt.show()
print('가격을 음수로 예측한 다이아몬드:', (y_pred < 0).sum(), '개')

✍️ **해석 쓰기** — 예측 가격이 음수인 다이아몬드가 생긴 이유는? 산점도에서 실제 가격이 비쌀수록 점들이 대각선에서 어느 쪽으로 벗어나는가?

> (여기에 한두 문장으로 적어 보세요)

✏️ 빈칸 채우기: 새 다이아몬드(1캐럿, 6.4×6.3×3.9mm)의 가격을 예측한다.

In [ ]:
new_diamond = pd.DataFrame({'carat': [___], 'x': [6.4], 'y': [6.3], 'z': [3.9]})
model.predict(new_diamond).round(0)

🔧 수정: 특성을 캐럿 하나만 쓰면 성능이 얼마나 달라질까? 특성 목록을 바꿔 실행해 본다.

In [ ]:
features = ['carat']          # ['carat', 'x', 'y', 'z'] 와 비교
m1 = LinearRegression().fit(X_train[features], y_train)
p1 = m1.predict(X_test[features])
print('특성:', features, '| MAE:', round(mean_absolute_error(y_test, p1)), '| R²:', round(r2_score(y_test, p1), 3))

---
# PART C. 지도학습 · 분류 — 다이아몬드 고가 / 일반 분류
**문제** 품질 정보(cut·color·clarity)만으로 고가/일반을 구분할 수 있을까? — 정답이 '종류'이므로 **분류**

✏️ 빈칸 채우기: 가격의 중앙값을 기준으로 '고가'/'일반' 정답 열을 만든다. (중앙값이라 거의 반반이 된다)

In [ ]:
median_price = dia['price'].median()
dia['price_label'] = np.where(dia['price'] >= median_price, '고가', '일반')
print('기준(중앙값):', median_price)
dia['price_label'].___

▶️ 교재처럼 무게가 같은 0.7캐럿만 골라, 품질 등급별 고가/일반 분포를 본다.

In [ ]:
d07 = dia[dia['carat'] == 0.7]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['cut', 'color', 'clarity']):
    sns.countplot(data=d07, x=col, hue='price_label', ax=ax)
    ax.set_title(f'{col}별 고가/일반 (0.7캐럿)')
plt.tight_layout()
plt.show()

🔮 예측 → 실행: 컷 등급이 가장 좋은 Ideal의 가격 중앙값은 가장 높을까? (예측: ___ )

In [ ]:
dia.groupby('cut', observed=True)[['price', 'carat']].median()

👀 **관찰** — 최고 등급 Ideal의 가격 중앙값이 오히려 가장 낮다! Ideal 다이아몬드는 **작은 것(캐럿이 낮은 것)이 많기 때문**이다.
가격을 가장 크게 좌우하는 것은 캐럿이다. 이 사실이 아래 분류 결과에 그대로 나타난다.

## 글자 범주를 숫자로 — 원-핫 인코딩(`get_dummies`)

▶️ 모델은 'Ideal', 'E' 같은 글자를 계산할 수 없다. 범주마다 0/1 열을 만들어 준다.

In [ ]:
X = dia[['cut', 'color', 'clarity']]
y = dia['price_label']
X_dummy = pd.get_dummies(X, drop_first=True)
print(X.shape, '→', X_dummy.shape)
X_dummy.head(3)

✏️ 빈칸 채우기: 학습용/평가용으로 나눈다. `stratify=y`는 두 묶음의 고가/일반 비율을 똑같이 맞춘다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_dummy, y, test_size=0.3, random_state=42, stratify=y)
y_train.value_counts(___).round(3)

▶️ 표준화 → 로지스틱 회귀(분류 모델) 학습 → 예측

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # 학습용으로 기준(평균·표준편차)을 정하고
X_test_scaled = scaler.transform(X_test)         # 평가용은 그 기준을 그대로 적용
model_lr = LogisticRegression(max_iter=2000)
model_lr.fit(X_train_scaled, y_train)
y_pred_lr = model_lr.predict(X_test_scaled)
y_pred_lr[:10]

✏️ 빈칸 채우기: 정확도와 혼동 행렬을 구한다.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

print('정확도:', round(___(y_test, y_pred_lr), 3))
cm = confusion_matrix(y_test, y_pred_lr, labels=['고가', '일반'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['예측: 고가', '예측: 일반'], yticklabels=['실제: 고가', '실제: 일반'])
plt.title('혼동 행렬 (품질 정보만 사용)')
plt.show()

▶️ 다른 분류 모델 맛보기 — 결정 트리 (교재 6장)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)
print('결정 트리 정확도:', round(accuracy_score(y_test, model_dt.predict(X_test)), 3))

🔧 수정: 특성에 캐럿(carat)을 더하면 정확도가 어떻게 달라질까?

In [ ]:
X2 = pd.get_dummies(dia[['carat', 'cut', 'color', 'clarity']], drop_first=True)
a_tr, a_te, b_tr, b_te = train_test_split(X2, y, test_size=0.3, random_state=42, stratify=y)
sc2 = StandardScaler()
m2 = LogisticRegression(max_iter=2000).fit(sc2.fit_transform(a_tr), b_tr)
print('캐럿 포함 정확도:', round(accuracy_score(b_te, m2.predict(sc2.transform(a_te))), 3))

✍️ **해석 쓰기** — 품질 정보만 쓴 모델(약 64%)과 캐럿을 더한 모델(약 98%)의 차이를 위의 '관찰'과 연결해 설명해 보자. 좋은 모델을 만들려면 무엇이 가장 중요할까?

> (여기에 한두 문장으로 적어 보세요)

---
# PART D. 비지도학습 · 군집 — 와인 데이터 군집화
**문제** 와인의 성분 정보만으로 비슷한 와인끼리 자동으로 묶을 수 있을까? — **정답(품종)을 주지 않고** 스스로 묶게 한다

▶️ 와인 178병의 성분 13가지를 불러온다. '품종' 열은 나중에 비교할 때만 쓰고, 모델에는 주지 않는다.

In [ ]:
wine = pd.read_csv(f'{DATA}/wine.csv')
X_wine = wine.drop(columns='품종')      # 정답을 뺀 성분 13가지
print(X_wine.shape)
X_wine.describe().round(2).loc[['mean', 'std', 'min', 'max'], ['alcohol', 'malic_acid', 'magnesium', 'proline']]

🔮 예측 → 실행: proline은 수백~천 단위, 다른 열은 한 자릿수다. 그대로 거리를 재면 어떤 열이 결과를 좌우할까? (예측: ___ ) — 표준화로 단위를 맞춘다.

In [ ]:
X_scaled = StandardScaler().fit_transform(X_wine)
pd.DataFrame(X_scaled, columns=X_wine.columns).describe().round(2).loc[['mean', 'std'], ['alcohol', 'malic_acid', 'magnesium', 'proline']]

✏️ 빈칸 채우기: K-평균(KMeans)으로 3개 군집을 만든다.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=___, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
pd.Series(labels).value_counts().sort_index()

▶️ 13차원 데이터를 PCA로 2차원에 눌러 담아 군집을 색으로 본다.

In [ ]:
from sklearn.decomposition import PCA

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
plt.figure(figsize=(5, 4))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', s=20)
plt.title('와인 군집화 결과 (PCA 2차원)')
plt.xlabel('주성분 1')
plt.ylabel('주성분 2')
plt.show()

✏️ 빈칸 채우기: 군집별 성분 평균을 비교해 각 군집의 특징을 찾는다. (3차시 groupby)

In [ ]:
profile = X_wine.copy()
profile['군집'] = labels
profile.___[['alcohol', 'color_intensity', 'flavanoids', 'proline']].mean().round(2)

👀 관찰: 몰래 숨겨 둔 실제 품종과 군집을 비교한다. 모델은 정답을 한 번도 보지 않았다!

In [ ]:
pd.crosstab(labels, wine['품종'], rownames=['군집'], colnames=['실제 품종'])

▶️ 군집 개수(K)는 몇 개가 좋을까? 엘보우(팔꿈치) 방법 — 꺾이는 지점을 찾는다.

In [ ]:
inertia = []
ks = range(1, 8)
for k in ks:
    inertia.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled).inertia_)
plt.plot(list(ks), inertia, marker='o')
plt.title('엘보우 방법으로 K 고르기')
plt.xlabel('군집 개수(K)')
plt.ylabel('inertia (군집 안 거리의 합)')
plt.show()

✍️ **해석 쓰기** — 군집 0·1·2에 '어떤 와인'인지 이름을 붙여 보자(예: 프롤린이 높은 진한 와인). 실제 품종과 비교했을 때 몇 병이 다르게 묶였나?

> (여기에 한두 문장으로 적어 보세요)

---
# PART E. ✏️ 직접 만들기 — 공공자전거 대여건수 예측 모델
4차시 히트맵에서 기온과 대여건수의 상관계수는 0.70이었다. 기온으로 대여건수를 예측하는 모델을 직접 만들어 보자.

E1. 데이터를 불러와 X(평균기온)와 y(대여건수)를 정한다.

In [ ]:
bike = pd.read_csv(f'{DATA}/bike_2025.csv')
Xb = bike[['평균기온']]
yb = bike['대여건수']
print(Xb.shape, yb.shape)

E2. 학습용 70% / 평가용 30%로 나누고 모델을 학습한다.

In [ ]:
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=___, random_state=42)
bike_model = LinearRegression().fit(Xb_tr, yb_tr)
print('기울기:', bike_model.coef_.round(0), '절편:', round(bike_model.intercept_))

E3. 평가용 데이터로 R²와 MAE를 구한다.

In [ ]:
pb = bike_model.predict(Xb_te)
print('R²:', round(___, 3), '| MAE:', round(mean_absolute_error(yb_te, pb)), '건')

E4. 🔧 수정: 특성에 '강수량'을 더하면 성능이 좋아질까?

In [ ]:
cols = ['평균기온', '강수량']
Xb2 = bike[___]
a, b, c, d = train_test_split(Xb2, yb, test_size=0.3, random_state=42)
m_b2 = LinearRegression().fit(a, c)
print('특성:', cols, '| R²:', round(r2_score(d, m_b2.predict(b)), 3))

E5. 나만의 날씨로 예측: 기온 25℃, 비가 안 온 날(강수량 0)의 대여건수는?

In [ ]:
cols = ['평균기온', '강수량']
m_b2 = LinearRegression().fit(bike[cols], yb)
m_b2.predict(pd.DataFrame({'평균기온': [25], '강수량': [___]})).round(0)

✍️ **해석 쓰기** — 기온만 쓴 모델과 강수량을 더한 모델의 R²를 비교해 보자. 4차시 산점도(기온 vs 대여건수)에서 기온이 아주 높은 날 대여가 줄어들던 모습을 떠올리면, 직선 모델의 한계는 무엇일까?

> (여기에 한두 문장으로 적어 보세요)

---
### 오늘의 정리
| | 지도학습 · 회귀 | 지도학습 · 분류 | 비지도학습 · 군집 |
|---|---|---|---|
| 정답(y) | 있음 · **숫자** | 있음 · **종류** | **없음** |
| 오늘의 예 | 아이스크림 판매량 · 다이아몬드 가격 | 다이아몬드 고가/일반 | 와인 묶기 |
| 모델 | `LinearRegression` | `LogisticRegression` · `DecisionTreeClassifier` | `KMeans` |
| 평가 | MAE · R² | 정확도 · 혼동 행렬 | 군집 특징 해석 · (정답이 있다면) 비교 |

**공통 흐름**: X·y 준비 → `train_test_split` → `model.fit(X_train, y_train)` → `model.predict(X_test)` → 평가
**기억할 것**: 학습 범위 밖 예측 주의 · 평가는 처음 보는 데이터로 · 글자 범주는 `get_dummies` · 단위가 다르면 표준화 · 좋은 특성(캐럿!)이 좋은 모델을 만든다